In [1]:
import pandas as pd
import openpyxl

In [2]:
df = pd.read_excel("/home/callum/sno_data/raw/zhang.xlsx", header = 0, sheet_name = "Class I PTMs")

peptides = pd.read_excel("/home/callum/sno_data/raw/zhang.xlsx", header = 0, sheet_name = "Modified Peptides")

print(df.head())
# FDR filter of 0.01 was applied to peptides - < 0.01 gave "high" confidence column score, so can filter by this 

  Protein Accession  Position Modification Name Target Amino Acid  \
0       AT1G01050.1        24           Phospho                 S   
1       AT1G01050.1        96   S-nitrosylation                 C   
2       AT1G01050.1       142   S-nitrosylation                 C   
3       AT1G01060.7       129           Phospho                 S   
4       AT1G01060.7       553           Phospho                 S   

   Site Probability Confidence            Peptide Sequence  \
0              99.7       High                     ILSSLSR   
1             100.0       High  TLCEDNDPIDVLVIMQEPVLPGCFLR   
2             100.0       High                IIAVCVDDPEYK   
3              98.7       High         LVSSASSSQLNQAFLDLEK   
4             100.0       High          ERLPQSFSPPQVAENVNR   

   Position in Peptide  # Modified PSMs  # Uncertain Modified PSMs  \
0                    3               11                          0   
1                    3                2                          0   
2 

In [3]:
# filter out non high confidence scores 
vc = df["Confidence"].nunique()
print(vc)
# they are all high - FDR < 0.01

print("number of sites total:",len(df))
# filter for SNO specifically
sno_only = df[df["Modification Name"] == "S-nitrosylation"]
print("number of SNO sites:",len(sno_only))

print(df.columns)

1
number of sites total: 12148
number of SNO sites: 5952
Index(['Protein Accession', 'Position', 'Modification Name',
       'Target Amino Acid', 'Site Probability', 'Confidence',
       'Peptide Sequence', 'Position in Peptide', '# Modified PSMs',
       '# Uncertain Modified PSMs', '# Unmodified PSMs', 'Protein Description',
       'Master', 'Motif', 'Unnamed: 14'],
      dtype='str')


In [4]:
import io, re, time
import pandas as pd
import requests

# --- edit these to match your table ---
ID, POS, AA, PEP = "Protein Accession", "Position", "Target Amino Acid", "Peptide Sequence"
# df must have ID (e.g. AT1G01050.1), POS (1-based), AA; PEP is optional
TAXID = 3702                                  # Arabidopsis thaliana
URL = "https://rest.uniprot.org/uniprotkb/stream"
AGI = re.compile(r"AT[1-5CM]G\d{5}", re.I)

work = sno_only.copy()
work["locus"] = work[ID].str.extract(f"({AGI.pattern})", flags=re.I)[0].str.upper()   # drop the .1
loci = work["locus"].dropna().unique().tolist()

import os


hits = pd.read_csv("~/arabadopsis_proteome/uniprotkb_proteome_UP000006548_2026_11_06.tsv", sep="\t")   # your TSV's name
print(len(hits), "entries;", hits["Gene Names"].fillna("").str.contains("AT[1-5CM]G").mean().round(2), "have an AT locus")



hits["locus"] = hits["Gene Names"].fillna("").apply(lambda s: sorted({m.upper() for m in AGI.findall(s)}))
hits = hits.explode("locus").dropna(subset=["locus"])
hits = hits[hits["locus"].isin(loci)]
hits["is_reviewed"] = hits["Reviewed"].eq("reviewed")

# every (row, candidate UniProt entry) pair for that locus
cand = work.reset_index(names="row").merge(
    hits[["locus", "Entry", "is_reviewed", "Length", "Sequence"]], on="locus", how="left")

def pep_ok(seq, pep):
    if not isinstance(seq, str) or not isinstance(pep, str):
        return False
    return "".join(c for c in pep if c.isalpha()) in seq

def res_ok(seq, pos, aa):
    return isinstance(seq, str) and 0 < int(pos) <= len(seq) and seq[int(pos)-1] == aa

cand["residue_ok"] = [res_ok(s, p, a) for s, p, a in zip(cand["Sequence"], cand[POS], cand[AA])]
cand["peptide_found"] = [pep_ok(s, p) for s, p in zip(cand["Sequence"], cand[PEP])] if PEP in cand else False
n_cand = cand.groupby("row")["Entry"].nunique()

# best entry per row: residue matches > peptide found > reviewed > longest
best = (cand.sort_values(["row", "residue_ok", "peptide_found", "is_reviewed", "Length"],
                         ascending=[True, False, False, False, False])
            .drop_duplicates("row").set_index("row"))
out = work.join(best[["Entry", "is_reviewed", "residue_ok", "peptide_found"]]
                .rename(columns={"Entry": "uniprot_id", "is_reviewed": "reviewed"}))
out["n_candidates"] = n_cand
out["window"] = [
    (s[max(0, int(p)-4):int(p)-1] + "[" + s[int(p)-1] + "]" + s[int(p):int(p)+3])
    if isinstance(s, str) and 0 < int(p) <= len(s) else None
    for s, p in zip(best["Sequence"].reindex(out.index), out[POS])]

print(f"Mapped {out['uniprot_id'].notna().sum()} / {len(out)} rows")
print(f"Residue matches: {out['residue_ok'].sum()} | peptide found: {out['peptide_found'].sum()}")
print("Unmapped loci:", out.loc[out['uniprot_id'].isna(), 'locus'].unique().tolist())
out

39144 entries; 0.0 have an AT locus
Mapped 5897 / 5952 rows
Residue matches: 5820 | peptide found: 5862
Unmapped loci: ['ATCG00020', 'ATCG00120', 'ATCG00130', 'ATCG00160', 'ATCG00180', 'ATCG00330', 'ATCG00340', 'ATCG00350', 'ATCG00420', 'ATCG00430', 'ATCG00470', 'ATCG00480', 'ATCG00490', 'ATCG00500', 'ATCG00640', 'ATCG00660', 'ATCG00740', 'ATCG00760', 'ATCG00770', 'ATCG00780', 'ATCG00800', 'ATCG00810', 'ATCG00830', 'ATCG01060', 'ATCG01090', 'ATCG01110', 'ATCG01130', 'ATCG01280', 'ATMG00090', 'ATMG00160', 'ATMG00510', 'ATMG00640']


,Protein Accession,Position,Modification Name,Target Amino Acid,Site Probability,Confidence,Peptide Sequence,Position in Peptide,# Modified PSMs,# Uncertain Modified PSMs,...,Master,Motif,Unnamed: 14,locus,uniprot_id,reviewed,residue_ok,peptide_found,n_candidates,window
1,AT1G01050.1,96,S-nitrosylation,C,100.0,High,TLCEDNDPIDVLVIMQEPVLPGCFLR,3,2,0,...,Master Protein,FVPRTLcEDNDPI,AT1G01050.1_96,AT1G01050,Q93V56,True,True,True,1,RTL[C]EDN
2,AT1G01050.1,142,S-nitrosylation,C,100.0,High,IIAVCVDDPEYK,5,29,0,...,Master Protein,DKIIAVcVDDPEY,AT1G01050.1_142,AT1G01050,Q93V56,True,True,True,1,IAV[C]VDD
5,AT1G01090.1,101,S-nitrosylation,C,100.0,High,SFEDMCAQMYYR,6,28,0,...,Master Protein,RSFEDMcAQMYYR,AT1G01090.1_101,AT1G01090,O24457,True,True,True,1,EDM[C]AQM
6,AT1G01090.1,317,S-nitrosylation,C,100.0,High,GEGPTLVECETYR,9,2,0,...,Master Protein,GPTLVEcETYRFR,AT1G01090.1_317,AT1G01090,O24457,True,True,True,1,LVE[C]ETY
7,AT1G01100.4,9,S-nitrosylation,C,100.0,High,STVGELACSYAVMILEDEGIAITADK,8,4,0,...,Master Protein,TVGELAcSYAVMI,AT1G01100.4_9,AT1G01100,Q8LCW9,True,True,True,2,ELA[C]SYA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12143,ATMG00160.1,198,S-nitrosylation,C,100.0,High,CDAVPGR,1,1,0,...,Master Protein,PSSGVKcDAVPGR,ATMG00160.1_198,ATMG00160,NaN,NaN,False,False,0,NaN
12144,ATMG00510.1,92,S-nitrosylation,C,100.0,High,LLNCEVPLR,4,2,0,...,Master Protein,VEKLLNcEVPLRA,ATMG00510.1_92,ATMG00510,NaN,NaN,False,False,0,NaN
12145,ATMG00640.1,100,S-nitrosylation,C,100.0,High,ICGTVVESLPMAR,2,5,0,...,Master Protein,RISLRIcGTVVES,ATMG00640.1_100,ATMG00640,NaN,NaN,False,False,0,NaN
12146,ATMG00640.1,125,S-nitrosylation,C,100.0,High,TVQALLCR,7,3,0,...,Master Protein,TVQALLcRNLNVK,ATMG00640.1_125,ATMG00640,NaN,NaN,False,False,0,NaN


In [7]:
df_final = out[["uniprot_id", "Position"]].dropna()
df_final["organism"] = "Arabidopsis thaliana"

# rename columns
df_final = df_final.rename(columns= {"uniprot_id": "id", "Position": "position"})

print(df_final.head())
proceed = input("happy?")
if proceed == "yes":
    df_final.to_csv("~/sno_data/processed/zhang/zhang_org.csv", index = False)

       id  position              organism
1  Q93V56        96  Arabidopsis thaliana
2  Q93V56       142  Arabidopsis thaliana
5  O24457       101  Arabidopsis thaliana
6  O24457       317  Arabidopsis thaliana
7  Q8LCW9         9  Arabidopsis thaliana


In [29]:
from pepmatch import Preprocessor, Matcher

# Preprocess a proteome (one-time step)
Preprocessor('/home/callum/human_proteome/proteome/human_proteome.fasta').preprocess(k=5)

# Search for exact matches
df = Matcher(
  query=['YLLDLHSYL', 'GLCTLVAML', 'FAKEPEPTIDE'],
  proteome_file='/home/callum/human_proteome/proteome/human_proteome.fasta',
  max_mismatches=0,
  k=5
).match()

print(df)

Preprocessing human_proteome with k=5...
Saved to ./human_proteome_5mers.pepidx
Searching 3 peptides against human_proteome (k=5, max_mismatches=0)...
shape: (3, 15)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ Query ID ┆ Query     ┆ Matched   ┆ Protein   ┆ … ┆ Index end ┆ Protein   ┆ Gene      ┆ SwissProt │
│ ---      ┆ Sequence  ┆ Sequence  ┆ ID        ┆   ┆ ---       ┆ Existence ┆ Priority  ┆ Reviewed  │
│ str      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ i64       ┆ Level     ┆ ---       ┆ ---       │
│          ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ ---       ┆ i64       ┆ bool      │
│          ┆           ┆           ┆           ┆   ┆           ┆ i64       ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1        ┆ YLLDLHSYL ┆ YLLDLHSYL ┆ O60337.2  ┆ … ┆ 570       ┆ 1         ┆ 1         ┆ true      │
│ 2        ┆ GLCTLVAML ┆ n